In [3]:
import numpy as np
import pandas as pd
import mlflow
import joblib
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Modeling")

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1785101030275, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1785101030275, lifecycle_stage='active', name='Modeling', tags={}, trace_location=None, workspace='default'>

In [4]:
mlflow.sklearn.autolog()

path = "../data/processed/Bank_Churn_clean.csv"
df = pd.read_csv(path)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
TARGET = "Exited"
X = df.drop(columns=[TARGET])
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(X, y , test_size = 0.2, stratify= y, random_state=42)

In [6]:

ohe = OneHotEncoder( handle_unknown='ignore')
cat_cols = ['Gender', 'Geography']

ohe.fit(X_train[cat_cols])
X_train_ohe = ohe.transform(X_train[cat_cols])
X_test_ohe = ohe.transform(X_test[cat_cols])

ohe_cols = ohe.get_feature_names_out(cat_cols)
X_train[ohe_cols] = pd.DataFrame(X_train_ohe.toarray(), columns=ohe_cols, index=X_train.index)
X_test[ohe_cols] = pd.DataFrame(X_test_ohe.toarray(), columns=ohe_cols, index=X_test.index)

X_train = X_train.drop(columns=cat_cols)
X_test = X_test.drop(columns=cat_cols)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
with mlflow.start_run():
    dt = DecisionTreeClassifier(max_depth=10,class_weight="balanced")
    dt.fit(X_train_scaled, y_train)

    y_pred = dt.predict(X_test_scaled)
    y_proba = dt.predict_proba(X_test_scaled)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("F1-Score:", f1_score(y_test, y_pred))

2026/07/27 14:11:32 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/07/27 14:11:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.env\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


Accuracy: 0.746
Recall: 0.6805896805896806
Precision: 0.42290076335877863
F1-Score: 0.5216572504708098
🏃 View run sedate-snake-417 at: http://localhost:5000/#/experiments/3/runs/69f5218045804060a87e0fdfa410fa70
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [8]:
with mlflow.start_run():
    rf = RandomForestClassifier(max_depth=10,class_weight="balanced")
    rf.fit(X_train_scaled, y_train)

    y_pred = rf.predict(X_test_scaled)
    y_proba = rf.predict_proba(X_test_scaled)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("F1-Score:", f1_score(y_test, y_pred))

2026/07/27 14:11:43 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/07/27 14:11:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.env\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


Accuracy: 0.8275
Recall: 0.7174447174447175
Precision: 0.5593869731800766
F1-Score: 0.628632938643703
🏃 View run legendary-donkey-423 at: http://localhost:5000/#/experiments/3/runs/775d2700328d496ca16a87a227f7f166
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [9]:
mlflow.xgboost.autolog()
xgb = XGBClassifier(max_depth= 5, n_estimators=20)
xgb.fit(X_train_scaled, y_train)

y_pred = xgb.predict(X_test_scaled)
y_proba = xgb.predict_proba(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))

2026/07/27 14:11:50 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '04edceee769a4378bc68412109e91f90', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow
2026/07/27 14:11:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run judicious-kit-166 at: http://localhost:5000/#/experiments/3/runs/04edceee769a4378bc68412109e91f90
🧪 View experiment at: http://localhost:5000/#/experiments/3
Accuracy: 0.865
Recall: 0.4668304668304668
Precision: 0.7818930041152263
F1-Score: 0.5846153846153846


In [10]:
joblib.dump({"model:rf"}, "../models/rf_best.pkl")

['../models/rf_best.pkl']

In [11]:
scores = cross_val_score(rf, X_train_scaled, y_train, cv=5) 

2026/07/27 14:12:00 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '887eb70191364c09b67adda576969fa3', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/07/27 14:12:00 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/07/27 14:12:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.env\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and w

🏃 View run delicate-tern-257 at: http://localhost:5000/#/experiments/3/runs/887eb70191364c09b67adda576969fa3
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/07/27 14:12:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.env\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/07/27 14:12:15 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '78531c991aec4f078eabd786d18e2dce', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/07/27 14:12:15 WARNING mlflow.sklearn: Failed to log training dataset information to MLfl

🏃 View run popular-loon-844 at: http://localhost:5000/#/experiments/3/runs/e9955a35b92f478c95cbafe96f58293b
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/07/27 14:12:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.env\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/07/27 14:12:22 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'add2b2b5f33f48b28dace7317c7bd119', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/07/27 14:12:22 WARNING mlflow.sklearn: Failed to log training dataset information to MLfl

🏃 View run skillful-mink-237 at: http://localhost:5000/#/experiments/3/runs/78531c991aec4f078eabd786d18e2dce
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/07/27 14:12:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.env\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/07/27 14:12:29 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'dea479d6d130476880620cabdbf59bdf', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/07/27 14:12:29 WARNING mlflow.sklearn: Failed to log training dataset information to MLfl

🏃 View run mercurial-robin-707 at: http://localhost:5000/#/experiments/3/runs/add2b2b5f33f48b28dace7317c7bd119
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/07/27 14:12:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\DELL\Desktop\Projet Data\Data Science\.env\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


🏃 View run burly-pug-580 at: http://localhost:5000/#/experiments/3/runs/dea479d6d130476880620cabdbf59bdf
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [12]:
scores.mean(), scores.std()

(np.float64(0.8240000000000001), np.float64(0.0064055639876594685))